# Task 2 — Wound Stage Classification: Colab Ablation Run

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone / update the repo

In [ ]:
import os

REPO_DIR = "/content/CanRollasThesisProject"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/canrollas/CanRollasThesisProject.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

## 3. Fetch the dataset

Unzips the dataset (skipped if already extracted), then finds the
extracted `wound-stage-classification` folder wherever it landed inside
the zip (its internal layout has drifted before — this searches for it
instead of hardcoding a path) and symlinks it into the repo.

In [ ]:
import glob
import os

ZIP_PATH = "/content/drive/MyDrive/thesis/wound-stage-classification.zip"
EXTRACT_DIR = "/content/data"
DATASET_NAME = "wound-stage-classification"
DATASET_DIR = f"{REPO_DIR}/datasets/{DATASET_NAME}"

if not glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    !unzip -q "{ZIP_PATH}" -d {EXTRACT_DIR}

candidates = glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True)
assert candidates, f"couldn't find an extracted '{DATASET_NAME}' folder under {EXTRACT_DIR}"
source = candidates[0]
print("dataset found at:", source)

os.makedirs(os.path.dirname(DATASET_DIR), exist_ok=True)
if os.path.islink(DATASET_DIR):
    os.remove(DATASET_DIR)
os.symlink(source, DATASET_DIR)
!ls -la {DATASET_DIR}

## 4. Install dependencies

In [ ]:
import os
os.chdir(f"{REPO_DIR}/task_2")
!pip install -q -r requirements.txt

## 5. Redirect checkpoints/results to Drive

So a killed/disconnected runtime doesn't lose completed-run history or
checkpoints — everything survives on Drive across sessions.

In [ ]:
import yaml

OUTPUT_DIR = "/content/drive/MyDrive/thesis/task_2_outputs"

with open("configs/base.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["checkpoint_dir"] = f"{OUTPUT_DIR}/checkpoints"
cfg["results_csv"] = f"{OUTPUT_DIR}/results/ablation_results.csv"
with open("configs/base.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("checkpoint_dir:", cfg["checkpoint_dir"])
print("results_csv:   ", cfg["results_csv"])

## 6. Resolve the encoder x head grid

`configs/grid.yaml` is the 13-encoder x 2-head (softmax vs CORN)
cross-product — this dry-runs every combination with a dummy batch to
fail loudly on a typo/incompatible encoder, and records each config's
real parameter count to `configs/resolved_grid.json`.

In [ ]:
!python build_grid.py

## 7. Run the ablation grid

`run_ablation.py` already skips completed `(config, fold)` rows,
resumes a killed fold from its last saved epoch, and isolates each
config's failures so one bad config can't stall the rest — but a
crash still kills the Python process itself. This loop restarts it
automatically so a crash doesn't need you to re-run the cell by hand.
Output is streamed line-by-line (`Popen` + `-u`) instead of
`subprocess.run()`, which buffers the whole thing and prints nothing
in Colab until the process exits. A full runtime disconnect kills the
loop too (Colab tears down the kernel); at that point just re-run this
cell — the run picks up from the last saved epoch either way.

In [ ]:
import subprocess
import time

cmd = ["python", "-u", "run_ablation.py", "--workers", "8"]
while True:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    returncode = proc.wait()
    if returncode == 0:
        print("grid finished")
        break
    print(f"run_ablation.py exited with code {returncode}, retrying in 10s...")
    time.sleep(10)

## 8. Preview the ablation summary

`run_ablation.py` rewrites `results/ablation_summary.csv` (mean±std per
config over completed folds) after every finished fold, so this can be
re-run anytime mid-grid to check progress — ranked by macro-F1, with
the ordinal-aware MAE/QWK columns alongside to compare softmax vs CORN.

In [ ]:
import pandas as pd

summary_path = f"{OUTPUT_DIR}/results/ablation_summary.csv"
df = pd.read_csv(summary_path)
df.sort_values("macro_f1_mean", ascending=False)